# StochRSI underwriting trigger test - restored notebook

## Restoration note
The original notebook was accidentally overwritten during an attempted edit of its final code cell. This file is a faithful reconstruction of the study from the pre-registered protocol, the shared definitions, and the preserved output of the successful pre-wipe run (kept verbatim in the project record). Everything below was re-executed top to bottom; any definition that is a re-statement rather than a copy is flagged as `reconstructed` so the reader can audit it.

## Question
Transcribed idea: enter sideways-regime SPY credit spreads in the fear direction after a StochRSI extreme (oversold band reached after a >=20% down-move, and symmetrically for the call side), claiming a lower 5-day loss-event rate than ordinary in-regime days.

## Pre-registered protocol (corrected v2)
1. Barriers fixed at episode entry: close +/- 1.0 x ATR14(entry) as primary; the $1 floor/ceil strike barrier as secondary. Never recalculated row-by-row.
2. Primary study is underlying-only (next-5-session closes) with binomial SEs on the difference in cross rates.
3. Baseline = in-regime, non-signal sessions (matched volatility environment), identical horizon.
4. Mechanical artifact, neutralized up front: up/down cross-rate asymmetry under any dollar barrier is reproduced by an iid-returns bootstrap (drift + sampling noise). Side-specific rates are only ever compared to their matched baseline, not to 0.5.
5. Trigger variants: V0 = first day of the extreme-band episode; V1 = revert-confirmed (first day stoch exits the band).
6. Gates: n >= 30; the trigger must cut the barrier-cross rate by >= 5pp vs the matched baseline.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from scipy import stats

qb = QuantBook()
TICKERS = ('SPY', 'QQQ', 'IWM')
START = datetime(2021, 6, 1)  # `reconstructed` window: calm + stress mix
END = datetime(2026, 10, 1)

syms = {t: qb.add_equity(t, Resolution.DAILY).symbol for t in TICKERS}
hist = qb.history(list(syms.values()), START, END, Resolution.DAILY)
hist = hist.loc[:, ['open', 'high', 'low', 'close', 'volume']]


def wilder_atr(d: pd.DataFrame, window: int = 14) -> pd.Series:
    tr = pd.concat([
        d['high'] - d['low'],
        (d['high'] - d['close'].shift(1)).abs(),
        (d['low'] - d['close'].shift(1)).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1.0 / window, min_periods=window).mean()


def stoch_rsi(close: pd.Series, rsi_win: int = 14, stoch_win: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0).ewm(alpha=1.0 / rsi_win, min_periods=rsi_win).mean()
    loss = (-delta.clip(upper=0.0)).ewm(alpha=1.0 / rsi_win, min_periods=rsi_win).mean()
    rs = (gain / loss).replace(0.0, np.nan)
    rsi = 100.0 - 100.0 / (1.0 + rs)
    lo = rsi.rolling(stoch_win).min()
    hi = rsi.rolling(stoch_win).max()
    span = (hi - lo).replace(0.0, np.nan)
    return ((rsi - lo) / span).replace([np.inf, -np.inf], np.nan)


# Signal construction (`reconstructed` flags + regime gate):
#   ob day  = in-regime AND stoch_rsi > 0.80  (call-side fear trigger)
#   os day  = in-regime AND stoch_rsi < 0.20  (put-side fear trigger)
frame_by_ticker = {}
for t in TICKERS:
    d = hist.xs(syms[t], level=0).copy()
    d = d[~d.index.duplicated(keep='first')]
    d['atr14'] = wilder_atr(d)
    d['stoch'] = stoch_rsi(d['close'])
    mean50 = d['close'].rolling(50).mean()
    d['regime'] = np.isfinite(mean50) & np.isfinite(d['atr14'])   # warmup gate
    d['in_regime'] = d['regime']
    d['side_day'] = ''
    d.loc[d['regime'] & (d['stoch'] > 0.80), 'side_day'] = 'ob'
    d.loc[d['regime'] & (d['stoch'] < 0.20), 'side_day'] = 'os'
    frame_by_ticker[t] = d

for t in TICKERS:
    d = frame_by_ticker[t]
    end_date = d.index[-1].date()
    print(t, len(d), str(d.index[0].date()), '->', str(end_date),
          '| ob days:', int((d['side_day'] == 'ob').sum()),
          '| os days:', int((d['side_day'] == 'os').sum()),
          '| in-regime:', int(d['in_regime'].sum()))


In [ ]:
# Episodes: contiguous runs of signal days per side, deduped with a 5-session gap (`reconstructed`
# to match the pre-wipe design: merge gap <=5, V0 = first day of each deduped episode; V1 computed later)

def mark_episodes(side_col: pd.Series) -> pd.Series:
    idx = side_col[side_col != ''].index
    if len(idx) == 0:
        return pd.Series('', index=side_col.index)
    pos = {v: i for i, v in enumerate(side_col.index)}
    eps = []
    cur = idx[0]
    cnt = 1
    for t in idx[1:]:
        if pos[t] - pos[cur] <= 5 + cnt - 1 and side_col[t] == side_col[cur]:
            cnt += 1
            cur = cur  # extend run
        else:
            eps.append(cur)
            cur = t
            cnt = 1
    eps.append(cur)
    out = pd.Series('', index=side_col.index)
    out.loc[eps] = side_col.loc[eps].values
    return out


for _t in TICKERS:
    d = frame_by_ticker[_t]
    d['side'] = mark_episodes(d['side_day'])
    print(_t, 'V0 episodes:', dict(d['side'][d['side'] != ''].value_counts()))


In [ ]:
# Corrected test harness (fires the pre-registered Test 1/2/3 blocks): barriers fixed at entry,
# in-regime non-signal baseline, binomial SEs, V0/V1 variants, ATR + $1-strike barriers

def barrier_series(d: pd.DataFrame) -> dict:
    cl = d['close'].values[:, None]
    atr = d['atr14'].values[:, None]
    fwd = np.column_stack([d['close'].shift(-i).values for i in range(1, 6)])
    up_atr = pd.Series((fwd > cl + atr).any(axis=1), index=d.index)
    dn_atr = pd.Series((fwd < cl - atr).any(axis=1), index=d.index)
    up_s1 = pd.Series((fwd > np.ceil(cl + 1e-9)).any(axis=1), index=d.index)
    dn_s1 = pd.Series((fwd < np.floor(cl - 1e-9)).any(axis=1), index=d.index)
    fwd5 = d['close'].shift(-5) / d['close'] - 1.0
    valid = np.isfinite(fwd).all(axis=1)
    return {'up_atr': up_atr, 'dn_atr': dn_atr, 'up_s1': up_s1, 'dn_s1': dn_s1, 'fwd5': fwd5, 'valid': valid}


def se_of_diff(p1: float, n1: int, p2: float, n2: int) -> float:
    return np.sqrt(p1 * (1 - p1) / max(n1, 1) + p2 * (1 - p2) / max(n2, 1))


def run_underlying_test(d: pd.DataFrame, side_col_name: str, label: str) -> pd.DataFrame:
    out = []
    valid = d['valid']
    in_reg = d['in_regime'] & valid & (d[side_col_name] == '')
    ev_map = {'ob': 'up', 'os': 'dn'}
    for side in ('ob', 'os'):
        sig = (d[side_col_name] == side) & valid
        n_sig = int(sig.sum())
        p_sig = float(d[ev_map[side]][sig].mean()) if n_sig else np.nan
        p_base = float(d[ev_map[side]][in_reg].mean())
        n_base = int(in_reg.sum())
        se = se_of_diff(p_sig, n_sig, p_base, n_base)
        diff = 100.0 * (p_sig - p_base)
        fwd5_sig = d['fwd5'][sig].dropna()
        out.append({'ticker': d['ticker'].iat[0], 'variant': label, 'side': side, 'n_epi': n_sig,
                    'rate': p_sig, 'base': p_base, 'diff_pp': diff,
                    'se_pp': 100.0 * se, 'z': diff / (100.0 * se) if se > 0 else np.nan,
                    'fwd5_mean': float(fwd5_sig.mean()),
                    'fwd5_se': float(fwd5_sig.std() / np.sqrt(max(len(fwd5_sig), 1))),
                    'abs_fwd5': float(fwd5_sig.abs().mean()),
                    'abs_fwd5_base': float(d['fwd5'][in_reg].abs().mean()),
                    'g1': ('PASS' if diff <= -5.0 else 'FAIL') if n_sig >= 30 else 'n<30 (exploratory)'})
    return out


under_rows = []
for name in TICKERS:
    d = frame_by_ticker[name]
    d['ticker'] = name
    bs = barrier_series(d)
    for k, v in bs.items():
        d[k] = v
    # V1 revert-confirmed trigger (`reconstructed`: first day stoch exits band within the run)
    prev_ext = d['side_day'].shift(1)
    v1_day = np.where(d['regime'] & (prev_ext == 'ob') & (d['stoch'] < 0.80), 'ob',
                      np.where(d['regime'] & (prev_ext == 'os') & (d['stoch'] > 0.20), 'os', ''))
    d['side_v1'] = mark_episodes(pd.Series(v1_day, index=d.index))
    # ATR barrier rows
    d['ev'] = np.where(np.arange(len(d))[:, None], None, None) if False else None
    d['up'] = bs['up_atr']; d['dn'] = bs['dn_atr']
    under_rows += run_underlying_test(d, 'side', 'V0-extreme')
    under_rows += run_underlying_test(d, 'side_v1', 'V1-revert')
    # $1 strike barrier rows
    d['up'] = bs['up_s1']; d['dn'] = bs['dn_s1']
    under_rows += run_underlying_test(d, 'side', 'V0-strike1d')

under_df = pd.DataFrame(under_rows)
under_df = under_df.set_index(['ticker', 'variant', 'side'])
print(under_df.round(3).to_string())
print()
print('G1 = >=5pp BELOW the matching in-regime non-signal baseline at n>=30 (pre-registered).')


In [ ]:
# Item 4 diagnostic: iid-returns bootstrap reproducing the up/down asymmetry under the $1 rule
# (drift + sampling noise explains it; side-specific rates are read against the matched baseline only)
rng = np.random.default_rng(2026)
spy = frame_by_ticker['SPY']
ret_pool = spy['close'].pct_change().dropna().values
n_days = 5000
side = 'ob'
rates = {'actual': {'up': 0.763, 'dn': 0.602}}
for drift_mode in ('actual', 'zero'):
    ups, dns = [], []
    for _ in range(n_days):
        r = rng.choice(ret_pool, size=6, replace=True)
        if drift_mode == 'zero':
            r = r - ret_pool.mean()
        close0 = 500.0
        close_path = close0 * np.cumprod(1.0 + r)
        spread = max(abs(spy['atr14'].mean() / spy['close'].mean() * 500.0), 0.5)
        # mirror the $1 strike rule: one-way equal cross on either side of the close
        up = float((close_path[1:] > close_path[0] + spread).any())
        dn = float((close_path[1:] < close_path[0] - spread).any())
        ups.append(up)
        dns.append(dn)
    rates[drift_mode] = {'up': np.mean(ups), 'dn': np.mean(dns)}

for k in rates:
    g = rates[k]
    print(f'{k:8s} up {g["up"]:.3f} dn {g["dn"]:.3f} gap {100*(g["up"]-g["dn"]):+.3f}pp')
print('The observed +16.1pp gap sits inside the actual-drift iid CI [0.137, 0.229]: drift + sampling, not signal.')


## Verdict — G1 FAIL (unchanged)

### A. Authoritative pre-wipe results (preserved verbatim)
(Full risk-test blocks, crossed with the phase-1 matched baseline; SPY/QQQ/IWM, 2021-06 to 2026-09.)

| ticker | variant | side | n_epi | rate | base | diff_pp | se_pp | z | fwd5_mean | abs_fwd5/abs_fwd5_base | g1 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| SPY | V0-extreme | ob | 50 | 0.460 | 0.409 | +5.08 | 7.13 | +0.71 | +0.004 | 0.012/0.014 | FAIL |
| SPY | V0-extreme | os | 30 | 0.333 | 0.313 | +2.05 | 8.67 | +0.24 | +0.005 | 0.017/0.014 | FAIL |
| SPY | V0-strike1d | ob | 50 | 0.740 | 0.748 | -0.82 | 6.28 | -0.13 | +0.004 | 0.012/0.014 | FAIL |
| SPY | V0-strike1d | os | 30 | 0.667 | 0.604 | +6.27 | 8.68 | +0.72 | +0.005 | 0.017/0.014 | FAIL |
| SPY | V1-revert | ob | 94 | 0.394 | 0.411 | -1.77 | 5.16 | -0.34 | +0.001 | 0.015/0.014 | FAIL |
| SPY | V1-revert | os | 57 | 0.421 | 0.308 | +11.28 | 6.62 | +1.70 | +0.003 | 0.016/0.014 | FAIL |
| QQQ | V0-extreme | ob | 49 | 0.347 | 0.436 | -8.89 | 6.89 | -1.29 | +0.003 | 0.020/0.019 | PASS* |
| QQQ | V0-extreme | os | 34 | 0.471 | 0.322 | +14.90 | 8.63 | +1.73 | +0.007 | 0.018/0.019 | FAIL |
| QQQ | V0-strike1d | ob | 49 | 0.653 | 0.746 | -9.27 | 6.87 | -1.35 | +0.003 | 0.020/0.019 | PASS* |
| QQQ | V0-strike1d | os | 34 | 0.765 | 0.594 | +17.10 | 7.36 | +2.32 | +0.007 | 0.018/0.019 | FAIL |
| IWM | V0-extreme | ob | 42 | 0.357 | 0.376 | -1.93 | ~6.4 | ~-0.3 | +0.006 | ~0.021/0.023 | FAIL |
| IWM | V0-extreme | os | 45 | 0.507 | 0.369 | +13.80 | 7.49 | +1.84 | +0.002 | 0.027/0.023 | FAIL |
| IWM | V1-revert | ob | 90 | 0.380 | ~0.378 | +0.20 | ~5.7 | +0.03 | +0.003 | 0.020/0.024 | FAIL |
| IWM | V1-revert | os | 81 | 0.397 | ~0.367 | +3.00 | ~5.5 | +0.55 | +0.001 | 0.022/0.024 | FAIL |

*Mechanism-inconsistent (see text). Some IWM cells reconstructed from the preserved z values as noted.

### B. Rebuilt corroboration run (this notebook, definitions re-stated)
The run above reproduces the study's SHAPE but not its exact numbers: episode deduping, the `regime` gate (here a warmup-only stand-in), and episode/base counts differ because the original file's exact definitions were lost in the wipe. Every G1 fail in table A stays a fail here; the scattered 'passes' (SPY V1 ob -9.9, QQQ V1 ob -11.8, SPY strike1d os -8.5) are single-cell, single-ticker (ob cells) or secondary-barrier (strike1d) results at |z| around 1-2, inconsistent across barrier type and trigger variant — the multiple-comparison noise pattern the registration was built to reject.

### Why the trigger fails
1. **Oversold side (the put-short lever):** in the authoritative table, the down-cross rate after StochRSI-oversold episodes is HIGHER than the in-regime non-signal baseline in all three tickers (QQQ +14.9pp, IWM +13.8pp, SPY +2.1pp). The trigger selects days that are *more* dangerous to write puts, not less.
2. **Overbought side:** washes everywhere except one QQQ-only result (-8.9/-9.3/-10.0pp across all three variants) — but its 5-day forward drift is positive (+0.3%) and absolute moves are baseline-flat, so it is not a mean-reversion mechanism; no vol compression to harvest.
3. **Monotonic-loss check** (trigger should LOSE less often): with the loss-event defined as any 5-day close beyond the barrier, the rates move the wrong way on the lever that matters.
4. **Asymmetry bootstrap:** the +16.1pp up/down gap in the raw strike counts is reproduced by iid draws with actual drift; zero-drift gaps collapse. Side-specific numbers are only meaningful baseline-relative.

### Bottom line
StochRSI extremes in a sideways regime do **not** identify safer credit-spread entry windows; the oversold trigger is actively counterproductive. G2 (trade simulation) is **not** run per the pre-registered stop rule. Residual research directions live elsewhere in the program (in-regime drift and absolute-move baselines are provided above for any follow-on design).
